# CGHの更新開始からSLM表示までの時間

検証用4f再生・伝搬・PNG保存を計測から除き、係数設計→場生成→8-bit CGH→SLMフレーム準備→任意の実機表示関数を計測します。辞書とSDK接続は事前準備です。

**標準は実機未接続。** その場合は配列の準備完了までを測り、転送/表示時間は未計測にします。実際の表示はメーカーSDK等へ接続した `DISPLAY_CALLBACK` が必要です。SDKの戻りだけでは表示済み扱いにしません。詳細は `LSH数字1_SLM表示時間.md`。


In [ ]:
import importlib
import json
from pathlib import Path
from dataclasses import asdict
import lsh_digit1_theory as theory
import lsh_digit1_proposed as proposed
import lsh_digit1_slm_timing as timing
theory = importlib.reload(theory)
proposed = importlib.reload(proposed)
timing = importlib.reload(timing)

# 同じ条件の直接法スケールを読む。シミュレーションは再実行しません。
reference_dir = Path('lsh_digit1_proposed_rank100_shiftaware_v2')
reference = json.loads((reference_dir / 'summary.json').read_text(encoding='utf-8'))
if reference.get('study') != 'digit1_direct_vs_projected_shared_core':
    raise ValueError('数字1の比較結果フォルダを指定してください。')
config_values = dict(reference['config'])
config_values['sensor_z_m'] = tuple(config_values['sensor_z_m'])
config = theory.Digit1Config(**config_values)
REFERENCE_SCALE = reference['reference_scale']
RANK = reference['rank']
MODE_SPACING_FACTOR = reference.get('mode_spacing_factor', 1)
METHOD = 'proposed'  # 直接法も計測する場合は'direct'へ変更
REPEATS = 5
WARMUP = 1
INCLUDE_COEFFICIENT_DESIGN = True  # Falseなら係数が準備済みの時点から計測

# 実機のSDK接続/初期化はここで行います。標準は接続なし。
DISPLAY_CALLBACK = None  # 実装した同期表示関数を指定
EXPECTED_SLM_SHAPE = None  # 実機接続時は必須：(高さ,幅)
FRAME_TRANSFORM = None  # 必要なら校正済み座標変換/LUT等を行う関数
# 表示関数の戻り値例：timing.DisplayReceipt('controller_confirmed', '完了確認方法')
# 2048×2048を違う実機サイズへ自動リサイズはしません。

suffix = '' if MODE_SPACING_FACTOR == 1 else f'_spacing{MODE_SPACING_FACTOR}'
library_path = Path(f'lsh_digit1_library_rank{RANK}{suffix}_shiftaware.npz')
output_dir = Path(f'lsh_digit1_slm_timing_outputs_{METHOD}_rank{RANK}_v1')
print('実機接続:', DISPLAY_CALLBACK is not None)
print('保存先:', output_dir.resolve())


In [ ]:
summary = timing.run_slm_timing(
    output_dir, config, reference_scale=REFERENCE_SCALE, rank=RANK,
    library_path=library_path, method=METHOD, repeats=REPEATS, warmup=WARMUP,
    include_coefficient_design=INCLUDE_COEFFICIENT_DESIGN,
    display_callback=DISPLAY_CALLBACK, expected_slm_shape=EXPECTED_SLM_SHAPE,
    frame_transform=FRAME_TRANSFORM, mode_spacing_factor=MODE_SPACING_FACTOR)
print('結果:', output_dir.resolve())
if not summary['hardware_connected']:
    print('実機未接続：SLMフレームの準備完了までの計測です。表示時間は未計測。')
for name, stats in summary['statistics_seconds'].items():
    if stats is None:
        print(name, ': 未計測')
    else:
        print(name, ': median=', round(stats['median']*1000, 3), 'ms',
              'min=', round(stats['min']*1000, 3), 'ms',
              'max=', round(stats['max']*1000, 3), 'ms')
print('事前準備[s]:', summary['preparation_seconds'])
print('完了確認:', [(s['confirmation_stage'], s['confirmation_detail']) for s in summary['samples']])


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(output_dir / 'last_cgh_8bit.png'), width=650))
print('元CGH:', (output_dir / 'last_cgh_8bit.png').resolve())
print('SDKへ渡すフレーム:', (output_dir / 'last_slm_frame_8bit.png').resolve())
print('全測定値:', (output_dir / 'timing_samples.csv').resolve())
# 再計測する場合はoutput_dirをv2等の新しい名前へ変更してください。
